## 7; Régularisation

In [ ]:
# Import des librairies supplémentaires
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import cross_val_score, train_test_split, KFold
from sklearn.metrics import mean_squared_error

In [ ]:
# Préparation des données pour la régularisation
# On utilise les mêmes variables que le modèle complet (sans la constante, sklearn l'ajoute)

X_reg = df[["Surface_m2", "Chambres", "Annee_construction", 
            "Distance_centre_km", "Etage", "Ascenseur",
            "Revenu_median_quartier", "Qualite_ecole", "Covid"]].values

Y_reg = df["Prix_milliers_euros"].values

var_names_reg = ["Surface_m2", "Chambres", "Annee_construction", 
                 "Distance_centre_km", "Etage", "Ascenseur",
                 "Revenu_median_quartier", "Qualite_ecole", "Covid"]

### Standardisation des variables

**Pourquoi standardiser ?** Ridge et Lasso pénalisent les coefficients selon leur magnitude. Sans standardisation, les variables avec grandes échelles auraient des coefficients artificiellement petits, donc moins pénalisés.

**Formule de standardisation :**

$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{\sigma_j}$$

Après standardisation : $\bar{z}_j = 0$ et $\sigma_{z_j} = 1$ pour chaque variable $j$.

In [ ]:

def standardiser(X):
    """
    Standardise les colonnes de X (moyenne=0, écart-type=1).
    Retourne : X_scaled, moyennes, écarts-types
    """
    moyennes = X.mean(axis=0)
    ecarts_types = X.std(axis=0, ddof=0)  # ddof=0 pour cohérence avec sklearn
    X_scaled = (X - moyennes) / ecarts_types
    return X_scaled, moyennes, ecarts_types

X_scaled, X_moyennes, X_stds = standardiser(X_reg)

print("Vérification de la standardisation :")
print(f"Moyennes : {X_scaled.mean(axis=0).round(10)}")
print(f"Écarts-types : {X_scaled.std(axis=0).round(4)}")




### 1. Régression Ridge - Évolution des coefficients

**Ridge** ajoute une pénalité L2 (somme des carrés des coefficients) à la fonction de coût MCO :

$$\hat{\beta}_{Ridge} = \arg\min_{\beta} \left\{ \sum_{i=1}^{n}(y_i - x_i'\beta)^2 + \lambda \sum_{j=1}^{p}\beta_j^2 \right\}$$

Ou en notation matricielle :

$$\hat{\beta}_{Ridge} = \arg\min_{\beta} \left\{ \|Y - X\beta\|^2 + \lambda\|\beta\|_2^2 \right\}$$

**Solution analytique (forme fermée) :**

$$\boxed{\hat{\beta}_{Ridge} = (X'X + \lambda I_p)^{-1}X'Y}$$

- Quand $\lambda = 0$ : on retrouve l'estimateur MCO
- Quand $\lambda \to \infty$ : tous les coefficients tendent vers 0

In [5]:
lambdas = np.logspace(-2, 4, 100)  # λ de 0.01 à 10000

# Stockage des coefficients pour chaque λ
ridge_coefs = []

for lam in lambdas:
    ridge = Ridge(alpha=lam, fit_intercept=True)
    ridge.fit(X_scaled, Y_reg)
    ridge_coefs.append(ridge.coef_)

ridge_coefs = np.array(ridge_coefs)

# Graphique
plt.figure(figsize=(12, 6))
for i in range(len(var_names_reg)):
    plt.plot(lambdas, ridge_coefs[:, i], label=var_names_reg[i], linewidth=2)

plt.xscale('log')
plt.xlabel('λ (paramètre de régularisation)', fontsize=12)
plt.ylabel('Valeur des coefficients', fontsize=12)
plt.title('Ridge : Évolution des coefficients en fonction de λ', fontsize=14)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'np' is not defined


**Analyse Ridge :**

On observe sur le graphique que :
- Pour λ faible (≈ 0.01), les coefficients sont proches des valeurs MCO
- Quand λ augmente, tous les coefficients sont progressivement "shrinkés" vers 0
- **Surface_m2** reste le coefficient le plus élevé même pour λ grand, confirmant son importance dans l'explication du prix
- **Distance_centre_km** a un coefficient négatif qui diminue en valeur absolue
- Contrairement à Lasso, Ridge ne met jamais un coefficient exactement à 0 :c'est une régularisation "douce"

### 2. Régression Lasso - Évolution des coefficients

**Lasso** (Least Absolute Shrinkage and Selection Operator) ajoute une pénalité L1 (somme des valeurs absolues) :

$$\hat{\beta}_{Lasso} = \arg\min_{\beta} \left\{ \sum_{i=1}^{n}(y_i - x_i'\beta)^2 + \lambda \sum_{j=1}^{p}|\beta_j| \right\}$$

Ou en notation matricielle :

$$\boxed{\hat{\beta}_{Lasso} = \arg\min_{\beta} \left\{ \|Y - X\beta\|^2 + \lambda\|\beta\|_1 \right\}}$$

**Différence fondamentale avec Ridge :**
- La pénalité L1 n'est **pas différentiable** en $\beta_j = 0$
- Cela crée une **discontinuité** qui permet de mettre certains coefficients **exactement à zéro**
- Lasso effectue donc simultanément **estimation** et **sélection de variables**

**Pas de forme fermée** → résolution par algorithme itératif (descente de coordonnées)

In [ ]:
lasso_coefs = []

for lam in lambdas:
    lasso = Lasso(alpha=lam, fit_intercept=True, max_iter=10000)
    lasso.fit(X_scaled, Y_reg)
    lasso_coefs.append(lasso.coef_)

lasso_coefs = np.array(lasso_coefs)

# Graphique
plt.figure(figsize=(12, 6))
for i in range(len(var_names_reg)):
    plt.plot(lambdas, lasso_coefs[:, i], label=var_names_reg[i], linewidth=2)

plt.xscale('log')
plt.xlabel('λ (paramètre de régularisation)', fontsize=12)
plt.ylabel('Valeur des coefficients', fontsize=12)
plt.title('Lasso : Évolution des coefficients en fonction de λ', fontsize=14)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tableau de sélection des variables selon λ
print("Lasso - Sélection de variables selon λ :")
print(f"{'λ':>10} | {'Coefs ≠ 0':>10} | Variables conservées")
print("-"*70)

for lam in [0.1, 1, 10, 50, 100, 500]:
    lasso = Lasso(alpha=lam, fit_intercept=True, max_iter=10000)
    lasso.fit(X_scaled, Y_reg)
    n_nonzero = np.sum(lasso.coef_ != 0)
    selected = [var_names_reg[i] for i in range(len(var_names_reg)) if lasso.coef_[i] != 0]
    print(f"{lam:>10.1f} | {n_nonzero:>10} | {', '.join(selected) if selected else 'Aucune'}")

**Analyse Lasso :**
Le comportement de Lasso est fondamentalement différent de Ridge :
- Pour λ faible, les coefficients sont proches de MCO
- Quand λ augmente, certains coefficients deviennent - Les variables disparaissent progressivement : d'abord les moins importantes (ex: Covid, Etage), puis les autres
- **Surface_m2** est la dernière variable à rester non-nulle, confirmant son rôle central
- À λ = 500, seules X variables restent sélectionnées

Cette propriété de parcimonie (sparsity) est la principale différence avec Ridge.

### 3. Validation croisée K-fold pour choisir λ optimal

Le paramètre $\lambda$ contrôle le **compromis biais-variance** :
- $\lambda$ trop petit → surapprentissage (variance élevée)
- $\lambda$ trop grand → sous-apprentissage (biais élevé)

**Validation croisée K-fold** : on divise les données en K parties égales, puis pour chaque valeur de $\lambda$ :

$$\text{CV}(\lambda) = \frac{1}{K}\sum_{k=1}^{K} MSE_k(\lambda)$$

où $MSE_k$ est l'erreur sur le fold $k$ avec un modèle entraîné sur les $K-1$ autres folds.

**Choix optimal :**

$$\lambda^* = \arg\min_{\lambda} \text{CV}(\lambda)$$

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Ridge CV
ridge_cv_mse = []
for lam in lambdas:
    ridge = Ridge(alpha=lam)
    scores = cross_val_score(ridge, X_scaled, Y_reg, cv=kf, scoring='neg_mean_squared_error')
    ridge_cv_mse.append(-scores.mean())

ridge_cv_mse = np.array(ridge_cv_mse)
lambda_opt_ridge = lambdas[np.argmin(ridge_cv_mse)]

print(f"Ridge - λ optimal : {lambda_opt_ridge:.4f}")
print(f"Ridge - RMSE minimal (CV) : {np.sqrt(np.min(ridge_cv_mse)):.4f}")

In [ ]:
# Lasso CV
lasso_cv_mse = []
for lam in lambdas:
    lasso = Lasso(alpha=lam, max_iter=10000)
    scores = cross_val_score(lasso, X_scaled, Y_reg, cv=kf, scoring='neg_mean_squared_error')
    lasso_cv_mse.append(-scores.mean())

lasso_cv_mse = np.array(lasso_cv_mse)
lambda_opt_lasso = lambdas[np.argmin(lasso_cv_mse)]

print(f"Lasso - λ optimal : {lambda_opt_lasso:.4f}")
print(f"Lasso - RMSE minimal (CV) : {np.sqrt(np.min(lasso_cv_mse)):.4f}")

In [ ]:
# Graphiques CV
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lambdas, np.sqrt(ridge_cv_mse), 'b-', linewidth=2)
axes[0].axvline(x=lambda_opt_ridge, color='r', linestyle='--', 
                label=f'λ optimal = {lambda_opt_ridge:.2f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('λ', fontsize=12)
axes[0].set_ylabel('RMSE (CV 10-fold)', fontsize=12)
axes[0].set_title('Ridge : Sélection de λ', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(lambdas, np.sqrt(lasso_cv_mse), 'g-', linewidth=2)
axes[1].axvline(x=lambda_opt_lasso, color='r', linestyle='--', 
                label=f'λ optimal = {lambda_opt_lasso:.2f}')
axes[1].set_xscale('log')
axes[1].set_xlabel('λ', fontsize=12)
axes[1].set_ylabel('RMSE (CV 10-fold)', fontsize=12)
axes[1].set_title('Lasso : Sélection de λ', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Coefficients avec λ optimal
ridge_opt = Ridge(alpha=lambda_opt_ridge).fit(X_scaled, Y_reg)
lasso_opt = Lasso(alpha=lambda_opt_lasso, max_iter=10000).fit(X_scaled, Y_reg)

print("Coefficients avec λ optimal :")
print(f"{'Variable':<25} {'Ridge':>12} {'Lasso':>12}")
print("-"*50)
for i, var in enumerate(var_names_reg):
    print(f"{var:<25} {ridge_opt.coef_[i]:>12.4f} {lasso_opt.coef_[i]:>12.4f}")

### 4. Comparaison OLS vs Ridge vs Lasso (Train 80% - Test 20%)

Pour évaluer la performance prédictive, on divise les données en :
- **Échantillon d'entraînement** (80%) : pour estimer les coefficients
- **Échantillon de test** (20%) : pour évaluer la qualité de prédiction

**Métrique d'évaluation :**

$$RMSE = \sqrt{\frac{1}{n_{test}}\sum_{i \in test}(y_i - \hat{y}_i)^2}$$

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, Y_reg, test_size=0.2, random_state=42
)

print(f"Taille train : {len(X_train)}, Taille test : {len(X_test)}")

In [ ]:
# Entraînement des 3 modèles
ols = LinearRegression().fit(X_train, y_train)
ridge_final = Ridge(alpha=lambda_opt_ridge).fit(X_train, y_train)
lasso_final = Lasso(alpha=lambda_opt_lasso, max_iter=10000).fit(X_train, y_train)

# Prédictions
y_pred_ols = ols.predict(X_test)
y_pred_ridge = ridge_final.predict(X_test)
y_pred_lasso = lasso_final.predict(X_test)

# RMSE
rmse_ols = np.sqrt(mean_squared_error(y_test, y_pred_ols))
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

In [ ]:
# Tableau récapitulatif
print("="*60)
print("COMPARAISON DES MODÈLES - Erreur sur échantillon TEST")
print("="*60)
n_lasso_vars = np.sum(lasso_final.coef_ != 0)

print(f"{'Modèle':<25} {'RMSE Test':>15} {'Nb variables':>15}")
print("-"*60)
print(f"{'OLS':<25} {rmse_ols:>15.4f} {len(var_names_reg):>15}")
print(f"{'Ridge (λ=' + f'{lambda_opt_ridge:.2f})':<25} {rmse_ridge:>15.4f} {len(var_names_reg):>15}")
print(f"{'Lasso (λ=' + f'{lambda_opt_lasso:.2f})':<25} {rmse_lasso:>15.4f} {n_lasso_vars:>15}")

In [ ]:
# Comparaison des coefficients
print("\nComparaison des coefficients (données standardisées) :")
print(f"{'Variable':<25} {'OLS':>10} {'Ridge':>10} {'Lasso':>10}")
print("-"*60)
for i, var in enumerate(var_names_reg):
    print(f"{var:<25} {ols.coef_[i]:>10.4f} {ridge_final.coef_[i]:>10.4f} {lasso_final.coef_[i]:>10.4f}")

In [ ]:
# Graphiques prédictions vs réelles
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models_plot = [('OLS', y_pred_ols, rmse_ols), 
               ('Ridge', y_pred_ridge, rmse_ridge), 
               ('Lasso', y_pred_lasso, rmse_lasso)]

for ax, (name, y_pred, rmse) in zip(axes, models_plot):
    ax.scatter(y_test, y_pred, alpha=0.6)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
            'r--', linewidth=2, label='Prédiction parfaite')
    ax.set_xlabel('Valeurs réelles', fontsize=11)
    ax.set_ylabel('Valeurs prédites', fontsize=11)
    ax.set_title(f'{name}\nRMSE = {rmse:.2f}', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Discussion : Pourquoi les écarts-types classiques ne sont pas valides après Lasso ?

Les écarts-types et tests classiques (t-test, IC) ne sont **PAS valides** après Lasso :

**1. Biais de sélection**

Lasso effectue simultanément estimation ET sélection de variables. Les formules classiques supposent un modèle **fixe**, choisi indépendamment des données. L'incertitude du choix du modèle n'est pas prise en compte.

**2. Distribution non-normale**

Les estimateurs MCO suivent : $\hat{\beta}_{MCO} \sim \mathcal{N}(\beta, \sigma^2(X'X)^{-1})$

Les estimateurs Lasso :
- Sont **biaisés** vers 0 (shrinkage)
- Ont une probabilité non-nulle d'être **exactement à 0**
- Ne suivent pas une loi normale → les tests t et IC gaussiens ne s'appliquent pas

**3. Discontinuité**

Une petite perturbation des données peut faire passer un coefficient de $\hat{\beta}_j \neq 0$ à $\hat{\beta}_j = 0$ (ou inversement).

**4. Sous-estimation de la variance**

Calculer la variance "comme si" les variables avaient été choisies *a priori* sous-estime fortement la vraie variance.

### Résumé : Ridge vs Lasso

| Critère | Ridge | Lasso |
|---------|-------|-------|
| Pénalité | $\lambda\sum_j\beta_j^2$ (L2) | $\lambda\sum_j\|\beta_j\|$ (L1) |
| Solution | Forme fermée | Algorithme itératif |
| Coefficients | Shrinkés vers 0 | Certains exactement = 0 |
| Sélection de variables | Non | Oui |
| Multicolinéarité | Gère bien | Sélectionne une variable parmi les corrélées |
| Interprétabilité | Tous les prédicteurs | Modèle parcimonieux |

## 8. Prévisions
### 8.1 Prédiction ponctuelle et intervalle de confiance

In [ ]:
# Caractéristiques de la maison à prédire
x0_dict = {
    "Surface_m2": 120,
    "Chambres": 3,
    "Annee_construction": 2015,
    "Distance_centre_km": 5,
    "Etage": 1,
    "Ascenseur": 1,  # Oui = 1
    "Revenu_median_quartier": 65,  # en milliers (65 000 €)
    "Qualite_ecole": 7,
    "Covid": 0  # Année vente 2023 → pas Covid
}

# Note : "Distance_universite" n'est pas dans notre modèle
print("Caractéristiques de la maison à prédire :")
for var, val in x0_dict.items():
    print(f"  {var}: {val}")

#### 1. Prédiction ponctuelle

In [ ]:
# Reconstruction de la matrice X complète (avec constante) pour le modèle OLS
# On reprend les mêmes variables que dans model_origin

X_full = np.column_stack((
    np.ones(N),
    df["Surface_m2"],
    df["Chambres"],
    df["Annee_construction"],
    df["Distance_centre_km"],
    df["Etage"],
    df["Ascenseur"],
    df["Revenu_median_quartier"],
    df["Qualite_ecole"],
    df["Covid"]
))

Y_full = df["Prix_milliers_euros"].values

# Estimation OLS
XtX_inv = np.linalg.inv(X_full.T @ X_full)
B_hat = XtX_inv @ X_full.T @ Y_full

# Vecteur x0 (avec constante)
x0 = np.array([
    1,  # constante
    x0_dict["Surface_m2"],
    x0_dict["Chambres"],
    x0_dict["Annee_construction"],
    x0_dict["Distance_centre_km"],
    x0_dict["Etage"],
    x0_dict["Ascenseur"],
    x0_dict["Revenu_median_quartier"],
    x0_dict["Qualite_ecole"],
    x0_dict["Covid"]
])

# Prédiction ponctuelle
y_hat_0 = x0 @ B_hat

print("="*60)
print("PRÉDICTION PONCTUELLE")
print("="*60)
print(f"\nPrix prédit : {y_hat_0:.2f} milliers d'euros")
print(f"             = {y_hat_0 * 1000:.0f} €")

#### 2. Intervalle de confiance à 95%

In [ ]:
# Calcul des résidus et estimation de σ²
Y_hat = X_full @ B_hat
U_hat = Y_full - Y_hat
k = X_full.shape[1]  # nombre de paramètres (10)

sigma2_hat = np.sum(U_hat**2) / (N - k)
sigma_hat = np.sqrt(sigma2_hat)

print(f"Estimation de σ : {sigma_hat:.4f} (milliers €)")

In [ ]:
# Variance de la prédiction
# Il y a 2 types d'intervalles :
# 
# 1) Intervalle de CONFIANCE pour E[Y|X=x0] :
#    Var(ŷ₀) = σ² · x₀'(X'X)⁻¹x₀
#    → Incertitude sur la MOYENNE conditionnelle
#
# 2) Intervalle de PRÉVISION pour Y₀ :
#    Var(Y₀ - ŷ₀) = σ² · (1 + x₀'(X'X)⁻¹x₀)
#    → Incertitude sur une NOUVELLE observation (inclut l'erreur ε₀)

# Terme quadratique
quad_term = x0 @ XtX_inv @ x0

# Variance pour l'intervalle de confiance (moyenne)
var_conf = sigma2_hat * quad_term
se_conf = np.sqrt(var_conf)

# Variance pour l'intervalle de prévision (nouvelle obs)
var_pred = sigma2_hat * (1 + quad_term)
se_pred = np.sqrt(var_pred)

# t critique à 95%
t_crit = stats.t.ppf(0.975, N - k)

# Intervalles
IC_conf = (y_hat_0 - t_crit * se_conf, y_hat_0 + t_crit * se_conf)
IC_pred = (y_hat_0 - t_crit * se_pred, y_hat_0 + t_crit * se_pred)

print("="*60)
print("INTERVALLES À 95%")
print("="*60)

print(f"\n1) Intervalle de CONFIANCE pour E[Y|X=x₀] :")
print(f"   [{IC_conf[0]:.2f} ; {IC_conf[1]:.2f}] milliers €")
print(f"   = [{IC_conf[0]*1000:.0f} € ; {IC_conf[1]*1000:.0f} €]")
print(f"   Largeur : {(IC_conf[1] - IC_conf[0])*1000:.0f} €")

print(f"\n2) Intervalle de PRÉVISION pour Y₀ :")
print(f"   [{IC_pred[0]:.2f} ; {IC_pred[1]:.2f}] milliers €")
print(f"   = [{IC_pred[0]*1000:.0f} € ; {IC_pred[1]*1000:.0f} €]")
print(f"   Largeur : {(IC_pred[1] - IC_pred[0])*1000:.0f} €")

In [ ]:
# Visualisation
fig, ax = plt.subplots(figsize=(10, 6))

# Point de prédiction
ax.scatter([y_hat_0], [0], s=200, c='red', zorder=5, label=f'Prédiction : {y_hat_0:.1f}k€')

# Intervalle de confiance
ax.hlines(y=0.1, xmin=IC_conf[0], xmax=IC_conf[1], colors='blue', linewidth=8, 
          label=f'IC 95% (moyenne) : [{IC_conf[0]:.1f} ; {IC_conf[1]:.1f}]')
ax.scatter([IC_conf[0], IC_conf[1]], [0.1, 0.1], s=100, c='blue', marker='|')

# Intervalle de prévision
ax.hlines(y=-0.1, xmin=IC_pred[0], xmax=IC_pred[1], colors='green', linewidth=8, 
          label=f'IP 95% (nouvelle obs) : [{IC_pred[0]:.1f} ; {IC_pred[1]:.1f}]')
ax.scatter([IC_pred[0], IC_pred[1]], [-0.1, -0.1], s=100, c='green', marker='|')

ax.set_ylim(-0.5, 0.5)
ax.set_xlabel('Prix (milliers €)', fontsize=12)
ax.set_title('Prédiction et intervalles à 95%', fontsize=14)
ax.legend(loc='upper right', fontsize=10)
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

#### 3. Discussion : Cette prédiction est-elle fiable ?

In [ ]:
#### 3. Discussion : Cette prédiction est-elle fiable ?

# ---------------------------------------------------------------------
# A. QUALITÉ GLOBALE DU MODÈLE
# ---------------------------------------------------------------------

# Calcul R² et R² ajusté
SS_res = np.sum(U_hat**2)
SS_tot = np.sum((Y_full - Y_full.mean())**2)
R2 = 1 - SS_res / SS_tot
R2_adj = 1 - (SS_res / (N - k)) / (SS_tot / (N - 1))

print("="*70)
print("A. QUALITÉ GLOBALE DU MODÈLE")
print("="*70)
print(f"R²        : {R2:.4f} ({R2*100:.1f}% de la variance expliquée)")
print(f"R² ajusté : {R2_adj:.4f}")
print(f"σ̂         : {sigma_hat:.2f} milliers € ({sigma_hat/y_hat_0*100:.1f}% du prix prédit)")

# ---------------------------------------------------------------------
# B. VÉRIFICATION DE L'EXTRAPOLATION UNIVARIÉE
# ---------------------------------------------------------------------

print("\n" + "="*70)
print("B. EXTRAPOLATION UNIVARIÉE")
print("="*70)
print(f"{'Variable':<25} {'Min':>10} {'x₀':>10} {'Max':>10} {'Statut':>12}")
print("-"*70)

var_names_check = ["Surface_m2", "Chambres", "Annee_construction", 
                   "Distance_centre_km", "Etage", "Ascenseur",
                   "Revenu_median_quartier", "Qualite_ecole"]

variables_hors_plage = []
for var in var_names_check:
    val = x0_dict[var]
    min_val = df[var].min()
    max_val = df[var].max()
    in_range = min_val <= val <= max_val
    status = "✓ OK" if in_range else "✗ HORS PLAGE"
    if not in_range:
        variables_hors_plage.append(var)
    print(f"{var:<25} {min_val:>10.1f} {val:>10.1f} {max_val:>10.1f} {status:>12}")

if len(variables_hors_plage) == 0:
    print("\n→ Toutes les variables sont dans la plage des données d'entraînement.")
else:
    print(f"\n→ ATTENTION : Variables hors plage : {', '.join(variables_hors_plage)}")

# ---------------------------------------------------------------------
# C. VÉRIFICATION DE L'EXTRAPOLATION MULTIVARIÉE (LEVERAGE)
# ---------------------------------------------------------------------

print("\n" + "="*70)
print("C. EXTRAPOLATION MULTIVARIÉE (LEVERAGE)")
print("="*70)

h_00 = x0 @ XtX_inv @ x0
leverage_seuil = 2 * k / N

print(f"Leverage h₀₀ = x₀'(X'X)⁻¹x₀ : {h_00:.6f}")
print(f"Seuil critique (2k/n)       : {leverage_seuil:.6f}")

if h_00 < leverage_seuil:
    print("→ h₀₀ < 2k/n : Pas d'extrapolation multivariée détectée.")
else:
    print("→ h₀₀ ≥ 2k/n : ATTENTION - Point atypique, extrapolation risquée.")

# ---------------------------------------------------------------------
# D. PRÉCISION DES INTERVALLES
# ---------------------------------------------------------------------

print("\n" + "="*70)
print("D. PRÉCISION DE LA PRÉDICTION")
print("="*70)

largeur_IC = IC_conf[1] - IC_conf[0]
largeur_IP = IC_pred[1] - IC_pred[0]

print(f"Prix prédit : {y_hat_0*1000:,.0f} €")
print(f"\nIntervalle de confiance (95%) : [{IC_conf[0]*1000:,.0f} € ; {IC_conf[1]*1000:,.0f} €]")
print(f"   Largeur : {largeur_IC*1000:,.0f} € ({largeur_IC/y_hat_0*100:.1f}% du prix)")
print(f"\nIntervalle de prévision (95%) : [{IC_pred[0]*1000:,.0f} € ; {IC_pred[1]*1000:,.0f} €]")
print(f"   Largeur : {largeur_IP*1000:,.0f} € ({largeur_IP/y_hat_0*100:.1f}% du prix)")

# ---------------------------------------------------------------------
# E. VERDICT FINAL
# ---------------------------------------------------------------------

print("\n" + "="*70)
print("E. VERDICT FINAL")
print("="*70)

problemes = []
if R2_adj < 0.6:
    problemes.append("R² ajusté faible")
if len(variables_hors_plage) > 0:
    problemes.append(f"extrapolation univariée")
if h_00 >= leverage_seuil:
    problemes.append("leverage élevé")
if largeur_IP / y_hat_0 > 0.25:
    problemes.append("intervalle de prévision large (>25%)")

if len(problemes) == 0:
    print("PRÉDICTION FIABLE")
    print(f"\nLe modèle explique {R2_adj*100:.1f}% de la variance (R² ajusté).")
    print(f"Le point x₀ n'est pas atypique (h₀₀ = {h_00:.4f} < {leverage_seuil:.4f}).")
    print(f"L'intervalle de prévision représente {largeur_IP/y_hat_0*100:.1f}% du prix prédit.")
    print(f"\n→ Prix estimé : {y_hat_0*1000:,.0f} €")
    print(f"→ Fourchette réaliste (95%) : {IC_pred[0]*1000:,.0f} € - {IC_pred[1]*1000:,.0f} €")
else:
    print("PRÉDICTION AVEC RÉSERVES")
    print(f"\nProblèmes détectés : {', '.join(problemes)}")
    print(f"\n→ Prix estimé : {y_hat_0*1000:,.0f} € (à interpréter avec prudence)")